# 초전도체 임계온도 예측 — EDA와 모델 학습 (Superconductivity)

- 데이터: 초전도 물질의 물성 요약 (21,263행 · 82열)
- 목표: 임계온도(critical_temp, K) 예측 — 회귀
- 흐름: 불러오기 → 학습 전 확인 → EDA → 학습 → 해석
- 참고: 데이터 소개 data_Superconductivity.txt

- 이 데이터의 특징: **파생변수가 이미 만들어져 있음**
  (원소 물성 8종 × 요약 10방식 = 80개 특징)
  → 전처리 함정이 적고, 많은 변수를 그대로 학습

## 1. 불러오기

- 일반 CSV · 전부 수치형 · 결측 없음

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("train.csv")
print(df.shape)          # (21263, 82)
df.head()

### 1-1. 82개 컬럼의 구조 이해

- number_of_elements: 구성 원소 수
- 80개 특징 = 물성 8종 × 요약 10방식
  - 물성: 원자량·이온화E·원자반경·밀도·전자친화도·융해열·열전도도·원자가
  - 요약: mean·wtd_mean·gmean·wtd_gmean·entropy·wtd_entropy·range·wtd_range·std·wtd_std
- critical_temp: 임계온도 (예측 대상)
- 개별 컬럼을 다 이해할 필요 없음 → "원소 물성의 통계 요약" 

In [ ]:
# 컬럼 구성 확인
print("첫 컬럼:", df.columns[0])
print("마지막:", df.columns[-1])
print("특징 수:", df.shape[1]-1)   # 타깃 제외 81개

## 2. 학습 전 확인

- 이 데이터는 이미 정제됨 → 확인할 것이 적다

### 2-1. 0은 실제 값 (위장결측 아님)

- 구성 원소가 1종이면(number_of_elements=1, 약 1.3%)
  range·std 등이 0 (분산이 없으므로)
- 이 0은 정상 → 결측으로 바꾸지 말 것 (Pump it Up과 반대)

In [ ]:
print("원소 1종인 행:", (df["number_of_elements"]==1).sum(),
      f"({(df['number_of_elements']==1).mean():.1%})")
print("그때 range_atomic_mass 평균:",
      df[df["number_of_elements"]==1]["range_atomic_mass"].mean())
# 0에 가까움 — 원소 1종이면 범위가 없는 게 당연

### 2-2. 타깃 분포 확인

- 임계온도가 낮은 물질이 대부분, 고온은 소수
- 치우침이 있으면 로그 변환 검토

In [ ]:
import matplotlib.pyplot as plt
print(df["critical_temp"].describe().round(2))

df["critical_temp"].hist(bins=50, figsize=(8,3))
plt.title("critical temperature (K)")
plt.show()

## 3. EDA

- 변수가 많으므로(80+) minimal 모드

In [ ]:
from data_profiling import ProfileReport

profile = ProfileReport(df, minimal=True, progress_bar=False)
profile.to_file("superconductivity_eda.html")   # 브라우저에서

### 3-1. 열전도도와 임계온도

- 물성 중 임계온도와 관련 깊은 것을 살펴봄
- 예: wtd_mean_ThermalConductivity(가중 평균 열전도도)

In [ ]:
col = "wtd_mean_ThermalConductivity"
if col in df.columns:
    df.plot.scatter(x=col, y="critical_temp",
                    alpha=0.2, figsize=(7,3))
    plt.show()

## 4. 학습

- 특징 81개를 그대로 사용 (AutoGluon이 처리)
- 치우친 타깃 → 원본과 로그 변환 비교 가능

In [ ]:
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    df, test_size=0.2, random_state=42)

predictor = TabularPredictor(
    label="critical_temp",
    eval_metric="root_mean_squared_error",
).fit(train_data, presets="medium_quality", time_limit=600)

In [ ]:
predictor.leaderboard(test_data)

## 5. 해석

In [ ]:
print(predictor.evaluate(test_data))

### 5-1. 변수 중요도

- 80개 요약 특징 중 무엇이 임계온도를 예측하는가
- 어떤 물성(열전도도·원자가 등)이 상위에 오는가
- 물리적으로 말이 되는지 판단

In [ ]:
predictor.feature_importance(test_data).head(15)

## 정리

- 파생변수가 이미 만들어진 데이터 (물성 8종 × 요약 10방식)
- 전처리 함정이 적음 (결측 0, 정제됨)
- 0은 실제 값(원소 1종) → 위장결측 아님
- 80개 특징을 AutoGluon이 그대로 처리
- 변수 중요도로 어떤 물성이 중요한지 해석

- 다른 데이터와 비교:
  Concrete는 파생변수를 사람이 만들었지만,
  이 데이터는 이미 만들어져 있음 → 변수 해석에 집중